In [3]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.model_selection import KFold, train_test_split
from sklearn.preprocessing import LabelEncoder
import joblib, os
os.makedirs('./models', exist_ok=True)

# 데이터 로드
train  = pd.read_csv('C:/Users/user/dacon/Smart_Logistics/data/train.csv')
test   = pd.read_csv('C:/Users/user/dacon/Smart_Logistics/data/test.csv')
layout = pd.read_csv('C:/Users/user/dacon/Smart_Logistics/data/layout_info.csv')

# layout merge
layout_clean = layout[['layout_id', 'layout_type', 'aisle_width_avg',
                        'intersection_count', 'one_way_ratio', 'pack_station_count',
                        'charger_count', 'layout_compactness', 'zone_dispersion',
                        'robot_total', 'floor_area_sqm', 'ceiling_height_m',
                        'fire_sprinkler_count', 'emergency_exit_count']]

train = train.merge(layout_clean, on='layout_id', how='left')
test  = test.merge(layout_clean, on='layout_id', how='left')

le = LabelEncoder()
train['layout_type'] = le.fit_transform(train['layout_type'].fillna('unknown'))
test['layout_type']  = le.transform(test['layout_type'].fillna('unknown'))

TARGET   = 'avg_delay_minutes_next_30m'
ID_COLS  = ['ID', 'layout_id', 'scenario_id']
feature_cols = [c for c in train.columns if c not in ID_COLS + [TARGET]]

from catboost import CatBoostRegressor

X = train[feature_cols]
y = train[TARGET]
X_tr, X_val, y_tr, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

model_cat = CatBoostRegressor(
    iterations            = 200000,
    learning_rate         = 0.03,
    depth                 = 8,
    loss_function         = 'MAE',
    eval_metric           = 'MAE',
    task_type             = 'GPU',
    random_seed           = 42,
    early_stopping_rounds = 300,
    verbose               = 10000,
)

model_cat.fit(
    X_tr, y_tr,
    eval_set=(X_val, y_val),
)

best_iter = model_cat.best_iteration_
print(f'Best iteration: {best_iter}')
print(f'Validation MAE: {np.mean(np.abs(y_val - model_cat.predict(X_val))):.4f}')

Default metric period is 5 because MAE is/are not implemented for GPU


0:	learn: 14.5894112	test: 14.6450575	best: 14.6450575 (0)	total: 4.09ms	remaining: 13m 38s
10000:	learn: 8.6018756	test: 8.7395850	best: 8.7395850 (10000)	total: 40.8s	remaining: 12m 54s
20000:	learn: 7.9736769	test: 8.2898444	best: 8.2898444 (20000)	total: 1m 24s	remaining: 12m 40s
30000:	learn: 7.4932019	test: 7.9784744	best: 7.9784744 (30000)	total: 2m 6s	remaining: 11m 55s
40000:	learn: 7.0942362	test: 7.7418750	best: 7.7418750 (40000)	total: 2m 48s	remaining: 11m 13s
50000:	learn: 6.7470625	test: 7.5532381	best: 7.5532381 (50000)	total: 3m 30s	remaining: 10m 30s
60000:	learn: 6.4402688	test: 7.4004437	best: 7.4004437 (60000)	total: 4m 12s	remaining: 9m 49s
70000:	learn: 6.1694025	test: 7.2739388	best: 7.2739388 (70000)	total: 4m 54s	remaining: 9m 6s
80000:	learn: 5.9282444	test: 7.1686531	best: 7.1686531 (80000)	total: 5m 35s	remaining: 8m 23s
90000:	learn: 5.7127500	test: 7.0790900	best: 7.0790900 (90000)	total: 6m 17s	remaining: 7m 41s
100000:	learn: 5.5171606	test: 7.0030819	b

In [4]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)
oof_preds_cat  = np.zeros(len(train))
test_preds_cat = np.zeros(len(test))

cat_params = {
    'iterations'  : 199999,
    'learning_rate': 0.03,
    'depth'        : 8,
    'loss_function': 'MAE',
    'eval_metric'  : 'MAE',
    'task_type'    : 'GPU',
    'random_seed'  : 42,
    'verbose'      : 0,
}

for fold, (tr_idx, val_idx) in enumerate(kf.split(train)):
    print(f'── Fold {fold+1} ──')
    X_tr,  y_tr  = X.iloc[tr_idx],  y.iloc[tr_idx]
    X_val, y_val = X.iloc[val_idx], y.iloc[val_idx]

    model = CatBoostRegressor(**cat_params)
    model.fit(X_tr, y_tr)

    joblib.dump(model, f'C:/Users/user/dacon/Smart_Logistics/models/cat_fold{fold+1}.pkl')

    oof_preds_cat[val_idx] = model.predict(X_val)
    test_preds_cat        += model.predict(test[feature_cols]) / 5

    fold_mae = np.mean(np.abs(y_val - model.predict(X_val)))
    print(f'Fold {fold+1} MAE: {fold_mae:.4f}')

oof_mae = np.mean(np.abs(y - oof_preds_cat))
print(f'\nOOF MAE: {oof_mae:.4f}')

── Fold 1 ──


Default metric period is 5 because MAE is/are not implemented for GPU


Fold 1 MAE: 6.5769
── Fold 2 ──


Default metric period is 5 because MAE is/are not implemented for GPU


Fold 2 MAE: 6.6613
── Fold 3 ──


...

Fold 3 MAE: 6.5338
── Fold 4 ──


Default metric period is 5 because MAE is/are not implemented for GPU


Fold 4 MAE: 6.6460
── Fold 5 ──


Default metric period is 5 because MAE is/are not implemented for GPU


Fold 5 MAE: 6.5976

OOF MAE: 6.6031


In [5]:
test_id = pd.read_csv('C:/Users/user/dacon/Smart_Logistics/data/test.csv')['ID']

submission = pd.DataFrame({
    'ID'                        : test_id,
    'avg_delay_minutes_next_30m': test_preds_cat
})

submission.to_csv('C:/Users/user/dacon/Smart_Logistics/results/submission_v15_cat_kfold.csv', index=False)
print(submission.head())

            ID  avg_delay_minutes_next_30m
0  TEST_000000                   14.073984
1  TEST_000001                   15.706629
2  TEST_000002                   20.430621
3  TEST_000003                   18.125423
4  TEST_000004                   17.528147
